# 1/2 — Synthesize realistic resistor bands and run real segmentation

**Run this notebook FIRST**, preferably on a T4 GPU. It restores the three original COCO datasets from the existing checksummed Drive ZIP, loads your trained **LR-ASPP or PP-LiteSeg segmentation model**, and builds a **segmentation-in-the-loop** reader dataset.

### Important behavior
- Every original image is passed through the segmenter before its body crop is saved. The segmenter is run again after synthesis, rotation and shear to produce the actual reader input.
- **Training sources only** receive three variants per resistor: two **6-band** readings and one randomly chosen 3–6-band reading. Each uses valid resistor-code grammar and a color palette and real TRAIN-only photographed band textures, preserves body texture/shading, and gets annotation boxes transformed with its geometric warp.
- Val/test remain **real original photos, no synthesis**, so evaluation does not inflate performance through augmented duplicates.
- A bad or missing segmentation is **rejected**, counted and reported by split. The second notebook refuses to train if real-image coverage falls below 70% on any split. Accuracy is conditional on segmentation success: check the coverage report as well.
- Synthetic band appearances are approximations, not genuine photographs; inspect the visual preview and add real 6-band photos when possible.

### Files, caching and dependencies
- Select `SEGMENTER_CHOICE = 'lraspp'` (default) or `'ppliteseg'` below. **The first run automatically downloads the SHA-256-verified trained model from GitHub Releases if the ONNX is missing from Google Drive**: LR-ASPP from release `ee` (513 MB ZIP), or PP-LiteSeg from `v5` (966 MB ZIP). The ONNX graph, its external `.onnx.data` weights, and metadata are saved together; subsequent runs reuse the complete model. If a previous notebook saved only the small `.onnx` graph, it repairs that cache automatically. Preprocessing adapts to the chosen model.
- Source dataset is reused from `MyDrive/resistor_model/r1-mobile1d-colab/`.
- Results and SHA-verified resumable ZIP shards persist in `MyDrive/resistor_model/mobile1d-segsynthetic-data/shards/`. Shards are regenerated when the segmentation model SHA or source dataset changes.
- This does **not** modify the original source datasets, Mobile1D model checkpoints, or other XL checkpoints.

Estimated runtime depends on source count, ONNX Runtime provider, GPU/CPU speed and rejected masks. Processing can take hours; individual shards are checkpointed on Drive. Re-run the notebook to resume after interruption.


In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
# Set to 'ppliteseg' to use the exported PP-LiteSeg model instead.
SEGMENTER_CHOICE = 'lraspp'  # 'lraspp' or 'ppliteseg'
if SEGMENTER_CHOICE not in ('lraspp', 'ppliteseg'):
    raise ValueError("SEGMENTER_CHOICE must be lraspp or ppliteseg")
SEGMENTER_PATHS = {
    'lraspp': '/content/drive/MyDrive/resistor_model/segmentation-v4-lraspp/resistor_segmenter_lraspp.onnx',
    'ppliteseg': '/content/drive/MyDrive/resistor_model/segmentation-pp-liteseg/resistor_segmenter_ppliteseg.onnx',
}
os.environ['MOBILE1D_SEGMENTER_ONNX'] = SEGMENTER_PATHS[SEGMENTER_CHOICE]
BRANCH = 'feat/mobile1d-colab-progress-drive-cache'
MODEL_REF = BRANCH
os.environ['MOBILE1D_GIT_REF'] = BRANCH
repo = Path('/content/mobile1d_segsynthetic_repo')
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,
    'https://github.com/Persie0/resistor_model.git',str(repo)],check=True)
sys.path.insert(0,str(repo/'src'))
print('[checkout]',subprocess.check_output(['git','-C',str(repo),'rev-parse','--short','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-q','onnxruntime-gpu'],check=True)
import importlib.util
spec=importlib.util.spec_from_file_location('r1_preparer',repo/'colab/train_rres_mobile1d_colab.py')
base=importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)
base.mount_drive()
base.clone_training_repo()
base.install_dependencies()
print('[preflight] validating synthesis, geometry and split-isolation tests')
base.run([sys.executable,'-m','pytest','-q','tests/test_mobile1d_segsynthetic.py','tests/test_segmentation_release.py'],
         cwd=base.MODEL_REPO,heartbeat_seconds=base.HEARTBEAT_SECONDS)
print('[stage 1/3] restore/repair trained segmentation model')
from resistor_model.tools.segmentation_release import ensure_segmentation_onnx
segmenter_path = ensure_segmentation_onnx(SEGMENTER_CHOICE)
os.environ['MOBILE1D_SEGMENTER_ONNX'] = str(segmenter_path)
from resistor_model.tools.mobile1d_segsynthetic import Segmenter,prepare
print('[stage 2/3] restore annotated source photographs')
base.prepare_dataset()
print('[stage 3/3] load trained '+SEGMENTER_CHOICE+' and synthesize segmenter crops')
segmenter=Segmenter(segmenter_path)
prepare(segmenter)
print('[segsynthetic] done. Run notebook 2 to train Mobile1D XL.')


## Preview generated examples (optional)
Run this cell after at least one shard is finished. It shows examples produced **after** the segmentation + alignment step; poor-looking synth samples should not be used for final model selection.


In [ ]:
import io, json, zipfile, matplotlib.pyplot as plt, numpy as np, cv2
from pathlib import Path
shards=sorted(Path('/content/drive/MyDrive/resistor_model/mobile1d-segsynthetic-data/shards').glob('*.zip'))
if not shards: raise FileNotFoundError('No completed synthetic dataset shard yet')
with zipfile.ZipFile(shards[0]) as archive:
    rows=[json.loads(x) for x in archive.read('rows.jsonl').decode().splitlines() if x]
    chosen=[next((row for row in rows if row['split']=='train' and bool(row['synthetic'])==kind),None) for kind in (False,True)]
    chosen=[r for r in chosen if r is not None]
    fig,axes=plt.subplots(len(chosen),1,figsize=(13,3*len(chosen)),squeeze=False)
    for row,axis in zip(chosen,axes.flat):
        image=cv2.imdecode(np.frombuffer(archive.read(row['image']),np.uint8),cv2.IMREAD_COLOR)
        axis.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        for b in row['resistors'][0]['bands']:
            x=(b['bbox'][0]+b['bbox'][2])/2
            axis.axvline(x,color='yellow',alpha=.65)
            axis.text(x,4,b['color'],rotation=90,color='white',fontsize=8,bbox={'facecolor':'black','alpha':.65})
        axis.set_title(('Synthetic' if row['synthetic'] else 'Real')+' segmented crop · '+row['split'])
        axis.axis('off')
    plt.tight_layout(); plt.show()
